# Functions

In [3]:
import json

def getflowers(petal_length: float = 4.7) -> str:
    """
    Fetches iris / flowers info mation.

    :param petal_length: the petal length to filter species by (using less than), defaults to 4.7
    :return: a count of each flower with the petal length
    :rtype: json string
    """
    from sqlalchemy import create_engine, text
    import urllib
    import configparser; config = configparser.RawConfigParser()
    config.read('keys//keys.ini')
    
    
    s = text("""SELECT Species, COUNT(Species) count_of_species FROM iris_data 
WHERE [petal.length] < """ + str(petal_length) + """
GROUP BY Species""")

    Server = config['keys']['AzureSQLServer']
    Username = config['keys']['AzureSQLUser']
    Password = config['keys']['AzureSQLPassword']
    Database = config['keys']['AzureSQLDatabase']
    
    odbc_conn = 'Driver={ODBC Driver 17 for SQL Server};Server=tcp:' + \
        Server + f';Database={Database};Uid={Username};Pwd={Password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    params = urllib.parse.quote_plus(odbc_conn)
    conn_str = 'mssql+pyodbc:///?odbc_connect={}'.format(params)
    engine = create_engine(conn_str)
    connection = engine.connect()
    result = connection.execute(s).fetchall()
    
    return json.dumps(str(result))


getflowers(petal_length = 3.5)


'"[(\'setosa\', 50), (\'versicolor\', 3)]"'

In [4]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import FunctionTool, ToolSet
import json

# --- Build the tool definitions and toolset once ---
user_functions = {getflowers}
functions = FunctionTool(functions=user_functions)
toolset = ToolSet()
toolset.add(functions)

# --- Create client & agent ---
project = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint="https://aifsweden.services.ai.azure.com/api/projects/AIFSwedenProject01"
)

with project:
    project.agents.enable_auto_function_calls(toolset)

    agent = project.agents.create_agent(
        model="gpt-4.1-mini",
        name="SQLAgentWithTools",
        instructions="You are a helpful assistant",
        tools=functions.definitions,  # OK to provide definitions here
    )

    # Create thread and message
    thread = project.agents.threads.create()
    project.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content="Tell me all flowers that have petal length below 4.6?"
    )

    # Create & process the run (auto function calls will execute fetch_weather)
    run = project.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id
        # Alternative if you didn't call enable_auto_function_calls:
        # , toolset=toolset
    )
    print(f"Run finished with status: {run.status}")

    # (Optional) fetch and print the latest assistant message
    messages = list(project.agents.messages.list(thread_id=thread.id))
    for m in reversed(messages):
        if m.role == "assistant":
            print(m.content[0].text.value)
            break

    # Cleanup if you want:
    project.agents.delete_agent(agent.id)

Run finished with status: RunStatus.COMPLETED
There are three flower species that have some members with petal lengths below 4.6:
- Setosa (50 flowers)
- Versicolor (36 flowers)
- Virginica (1 flower)

If you would like detailed information about these flowers, please let me know!


# NL2SQL

In [5]:
def get_schema_text():
    import os
    import urllib
    import configparser
    from sqlalchemy import create_engine, text

    # Load config
    config = configparser.RawConfigParser()
    config.read(os.path.join('keys', 'keys.ini'))

    # Connection details
    Server = config['keys']['AzureSQLServer']
    Username = config['keys']['AzureSQLUser']
    Password = config['keys']['AzureSQLPassword']
    Database = config['keys']['AzureSQLDatabase']

    # ODBC connection string (URL-encoded)
    odbc_str = (
        f"Driver={{ODBC Driver 17 for SQL Server}};"
        f"Server=tcp:{Server};"
        f"Database={Database};"
        f"Uid={Username};"
        f"Pwd={Password};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Connection Timeout=30;"
    )
    odbc_conn = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"

    # Create engine
    engine = create_engine(odbc_conn)

    # Query schema
    query = text("""
        SELECT
            s.name AS SchemaName,
            t.name AS TableName,
            c.name AS ColumnName,
            ty.name AS DataType,
            c.max_length,
            c.is_nullable
        FROM sys.schemas s
        JOIN sys.tables t ON t.schema_id = s.schema_id
        JOIN sys.columns c ON c.object_id = t.object_id
        JOIN sys.types ty ON c.user_type_id = ty.user_type_id
        WHERE s.name = 'SalesLT'
        ORDER BY s.name, t.name, c.column_id;
    """)

    with engine.connect() as conn:
        result = conn.execute(query).fetchall()

    # Format schema text
    schema_text = "Database Schema:\n"
    current_table = None
    for schema, table, column, dtype, max_len, nullable in result:
        if table != current_table:
            schema_text += f"\nSchema: {schema}, Table: {table}\n"
            current_table = table
        schema_text += f"  - {column} ({dtype}, max_length={max_len}, nullable={nullable})\n"

    return schema_text

In [6]:
get_schema_text()

'Database Schema:\n\nSchema: SalesLT, Table: Address\n  - AddressID (int, max_length=4, nullable=False)\n  - AddressLine1 (nvarchar, max_length=120, nullable=False)\n  - AddressLine2 (nvarchar, max_length=120, nullable=True)\n  - City (nvarchar, max_length=60, nullable=False)\n  - StateProvince (Name, max_length=100, nullable=False)\n  - CountryRegion (Name, max_length=100, nullable=False)\n  - PostalCode (nvarchar, max_length=30, nullable=False)\n  - rowguid (uniqueidentifier, max_length=16, nullable=False)\n  - ModifiedDate (datetime, max_length=8, nullable=False)\n\nSchema: SalesLT, Table: Customer\n  - CustomerID (int, max_length=4, nullable=False)\n  - NameStyle (NameStyle, max_length=1, nullable=False)\n  - Title (nvarchar, max_length=16, nullable=True)\n  - FirstName (Name, max_length=100, nullable=False)\n  - MiddleName (Name, max_length=100, nullable=True)\n  - LastName (Name, max_length=100, nullable=False)\n  - Suffix (nvarchar, max_length=20, nullable=True)\n  - CompanyName

In [7]:
def runSQL(SQLText: str = "SELECT 1,2,3") -> str:
    """
    Runs SQL against the database.

    :param SQLText: the SQL to run
    :return: a result set as json string
    :rtype: json string
    """
    from sqlalchemy import create_engine, text
    import urllib
    import configparser; config = configparser.RawConfigParser()
    config.read('keys//keys.ini')
    
    s = text(SQLText)

    Server = config['keys']['AzureSQLServer']
    Username = config['keys']['AzureSQLUser']
    Password = config['keys']['AzureSQLPassword']
    Database = config['keys']['AzureSQLDatabase']
    
    odbc_conn = 'Driver={ODBC Driver 17 for SQL Server};Server=tcp:' + \
        Server + f';Database={Database};Uid={Username};Pwd={Password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    params = urllib.parse.quote_plus(odbc_conn)
    conn_str = 'mssql+pyodbc:///?odbc_connect={}'.format(params)
    engine = create_engine(conn_str)
    connection = engine.connect()
    result = connection.execute(s).fetchall()
    
    return json.dumps(str(result))


runSQL()

'"[(1, 2, 3)]"'

In [8]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import FunctionTool, ToolSet
import json

dbSchema = get_schema_text()

user_functions = {runSQL}
functions = FunctionTool(functions=user_functions)
toolset = ToolSet()
toolset.add(functions)

# --- Create client & agent ---
project = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint="https://aifsweden.services.ai.azure.com/api/projects/AIFSwedenProject01"
)

with project:
    project.agents.enable_auto_function_calls(toolset)

    agent = project.agents.create_agent(
        model="gpt-4.1-mini",
        name="SQLAgentWithTools",
        instructions="You create and run SQL against the database. The database schema is as follows:\n" + dbSchema + "\nUse the runSQL function to execute SQL queries. Always show the SQL statment at the top of the results",
        tools=functions.definitions,  # OK to provide definitions here
    )

    # Create thread and message
    thread = project.agents.threads.create()
    project.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content="Tell me how many of each produce have been sold"
    )

    # Create & process the run (auto function calls will execute fetch_weather)
    run = project.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id
        # Alternative if you didn't call enable_auto_function_calls:
        # , toolset=toolset
    )
    print(f"Run finished with status: {run.status}")

    # (Optional) fetch and print the latest assistant message
    messages = list(project.agents.messages.list(thread_id=thread.id))
    for m in reversed(messages):
        if m.role == "assistant":
            print(m.content[0].text.value)
            break

    # Cleanup if you want:
    project.agents.delete_agent(agent.id)

Run finished with status: RunStatus.COMPLETED
SQL statement:
```sql
SELECT p.Name AS ProductName, SUM(sod.OrderQty) AS TotalQuantitySold
FROM SalesLT.SalesOrderDetail sod
JOIN SalesLT.Product p ON sod.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY TotalQuantitySold DESC;
```

Here is the number of each product sold (top entries):
- Classic Vest, S: 87
- Short-Sleeve Classic Jersey, XL: 57
- Bike Wash - Dissolver: 55
- Water Bottle - 30 oz.: 54
- AWC Logo Cap: 52
- Long-Sleeve Logo Jersey, L: 51
- Sport-100 Helmet, Black: 51
- Short-Sleeve Classic Jersey, L: 51
- Racing Socks, L: 51
- Hydration Pack - 70 oz.: 50

If you want to see more or results summarized differently, let me know!


# Local Secure Databases with local LLMs

In [9]:
from foundry_local import FoundryLocalManager

FoundryLocalManager().list_cached_models()

[FoundryModelInfo(alias=phi-3.5-mini, id=Phi-3.5-mini-instruct-generic-cpu, runtime=cpu, file_size=2590 MB, license=MIT),
 FoundryModelInfo(alias=phi-3.5-mini, id=Phi-3.5-mini-instruct-generic-gpu, runtime=webgpu, file_size=2211 MB, license=MIT)]

In [10]:
import openai
from foundry_local import FoundryLocalManager

alias = "Phi-3.5-mini-instruct-generic-gpu"

# Create a FoundryLocalManager instance. This will start the Foundry 
# Local service if it is not already running and load the specified model.
manager = FoundryLocalManager(alias)

# The remaining code us es the OpenAI Python SDK to interact with the local model.
# Configure the client to use the local Foundry service
client = openai.OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key,  # API key is not required for local usage
)

# Set the model to use and generate a streaming response
stream = client.chat.completions.create(
    model=manager.get_model_info(alias).id,
    messages=[{"role": "user", "content": "Using the schema below give me the SQL query to tell me how many of each produce have been sold. This is the database schema:" + get_schema_text()}],
    stream=True,
    max_tokens=4000,
    temperature=0.7,
    top_p=0.9,
)

# Print the streaming response
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

 To find out how many of each produce have been sold, we need to join the `SalesOrderDetail` table with the `Product` table on the `ProductID` column. Then, we can group the results by the `Name` of the produce and count the number of sales for each produce item. Here's the SQL query to achieve this:

```sql
SELECT P.Name, COUNT(SOD.SalesOrderDetailID) AS NumberOfSales
FROM SalesOrderDetail SOD
JOIN Product P ON SOD.ProductID = P.ProductID
WHERE P.Name LIKE '%produce%'
GROUP BY P.Name
ORDER BY NumberOfSales DESC;
```

This query will return a result set with two columns: `Name` and `NumberOfSales`. The `Name` column will contain the name of each produce item, and the `NumberOfSales` column will contain the count of sales for that produce item. The result set will be ordered in descending order of `NumberOfSales`, so the produce item with the most sales will appear first.

Note that this query assumes that the `Name` column in the `Product` table contains the word "produce" for all rele